# Data Cleaning – Interview Problems

**Learning Objectives:**
- Handle duplicates and detect outliers (IQR / z-score)
- Clean inconsistent categorical values
- Resolve date parsing issues
- Merge datasets with mismatched schemas
- Handle multiple delimiters and messy formats
- Coerce types and engineer features from datetime
- Bin continuous variables and encode categoricals
- Prepare data end-to-end for an ML pipeline

## 1. Setup – A Truly Messy Dataset

This dataset simulates real-world chaos: missing values, inconsistent formatting, outliers, and more.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

messy = pd.DataFrame({
    "ID": [1, 2, 2, 3, 4, 5, 5, 6, 7, 8, 9, 10],
    "Name": ["Alice", "Bob", "Bob", "Charlie", "Diana", "Eve", "Eve",
             "Frank", "Grace", "Henry", "Ivy", "Jack"],
    "Age": [25, 30, 30, 35, np.nan, 29, 29, -5, 42, 31, 27, 38],
    "Salary": [75000, 68000, 68000, 82000, 72000, 95000, 95000,
                65000, np.nan, 70000, 72000, 250000],  # 250K is an outlier
    "Join_Date": ["2022-06-15", "01/20/2023", "01/20/2023", "2021-03-10",
                  "2022-11-01", "2024/02/28", "2024-02-28", "2023-07-04",
                  "2022-12-01", "hello", "2025-01-15", "2023-09-10"],
    "Department": ["IT", "HR", "HR", "IT", "Finance", "hr", "HR",
                    "IT", "FINANCE", "IT", np.nan, "Finance"],
    "Rating": ["4.5", "3", "3", "5", "4", "3.8", "3.8",
               "4.2", "4.8", "3.5", "4", "4.6"]
})
print("Original messy dataset:")
print(messy)

## 2. Handling Duplicates

Identify and remove duplicate rows, then check for duplicates based on key columns.

In [ ]:
print("Duplicate rows:", messy.duplicated().sum())
print("Duplicates (subset=ID):", messy.duplicated(subset="ID").sum())

# Remove exact duplicates, keep first
clean = messy.drop_duplicates()
print(f"After removing exact duplicates: {clean.shape[0]} rows left")

# For ID duplicates, keep last occurrence
clean = clean.drop_duplicates(subset="ID", keep="last")
print(f"After removing ID duplicates: {clean.shape[0]} rows left")

## 3. Handling Outliers (IQR and Z-Score)

In [ ]:
# IQR method
Q1 = clean["Salary"].quantile(0.25)
Q3 = clean["Salary"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_iqr = clean[(clean["Salary"] < lower_bound) | (clean["Salary"] > upper_bound)]
print(f"IQR bounds: {lower_bound:.0f} – {upper_bound:.0f}")
print(f"Outliers (IQR):\n{outliers_iqr[['Name', 'Salary']]}")

# Z-score method
from scipy import stats
z_scores = np.abs(stats.zscore(clean["Salary"].dropna()))
outliers_z = clean.iloc[np.where(z_scores > 3)[0]]
print(f"Outliers (z-score > 3):\n{outliers_z[['Name', 'Salary']]}")

In [ ]:
# Cap the outlier instead of removing
clean.loc[clean["Salary"] > upper_bound, "Salary"] = upper_bound
print("Salary after capping:")
print(clean["Salary"].describe())

## 4. Cleaning Inconsistent Categorical Values

Standardize case, strip whitespace, and map synonyms.

In [ ]:
print("Original departments:", clean["Department"].unique())

# Standardize and map
# Ensure Department is string first, then strip/title
clean["Department"] = clean["Department"].astype(str).str.strip().str.title()
clean["Department"] = clean["Department"].replace("Hr", "HR")
clean["Department"] = clean["Department"].replace("It", "IT")
clean["Department"] = clean["Department"].replace("Finance", "Finance")
clean.loc[clean["Department"] == "Nan", "Department"] = "Unknown"

print("Cleaned departments:", clean["Department"].unique())

## 5. Date Parsing Issues

Handle multiple date formats and errors.

In [ ]:
# Attempt to parse all date formats
clean["Join_Date_Parsed"] = pd.to_datetime(clean["Join_Date"], errors="coerce")
print("Parsed dates:")
print(clean[["Join_Date", "Join_Date_Parsed"]])

# Rows where parsing failed
failed = clean[clean["Join_Date_Parsed"].isna()]
print(f"Failed to parse {len(failed)} date(s):")
print(failed[["Name", "Join_Date"]])

## 6. Type Coercion

Ensure numeric columns are actually numeric.

In [ ]:
# Rating is stored as string – convert to float
print("Rating dtype:", clean["Rating"].dtype)
clean["Rating"] = pd.to_numeric(clean["Rating"], errors="coerce")
print("After conversion:", clean["Rating"].dtype)

# Age has a negative value – treat as missing
clean.loc[clean["Age"] < 0, "Age"] = np.nan
print("Age after fixing negatives:")
print(clean["Age"].describe())

## 7. Feature Engineering from Dates

Extract year, month, day of week, and tenure.

In [ ]:
today = pd.Timestamp("2025-12-31")
clean["Year"] = clean["Join_Date_Parsed"].dt.year
clean["Month"] = clean["Join_Date_Parsed"].dt.month
clean["DayOfWeek"] = clean["Join_Date_Parsed"].dt.dayofweek  # 0=Monday
clean["Tenure_Days"] = (today - clean["Join_Date_Parsed"]).dt.days

print(clean[["Name", "Join_Date_Parsed", "Year", "Month", "DayOfWeek", "Tenure_Days"]])

## 8. Binning Continuous Variables

Convert Age and Salary into categorical bins.

In [ ]:
# Age bins
clean["AgeGroup"] = pd.cut(clean["Age"], bins=[0, 25, 35, 45, 100],
                            labels=["<25", "25-35", "35-45", "45+"])

# Salary bins: quartile-based
clean["SalaryBand"] = pd.qcut(clean["Salary"].rank(method="first"), q=4,
                                labels=["Low", "Medium", "High", "Very High"])

print(clean[["Name", "Age", "AgeGroup", "Salary", "SalaryBand"]])

## 9. Encoding Categorical Variables

Prepare data for ML: one-hot encode and label encode.

In [ ]:
# One-hot encode Department
dept_dummies = pd.get_dummies(clean["Department"], prefix="Dept")

# Label encode AgeGroup
age_group_map = {"<25": 0, "25-35": 1, "35-45": 2, "45+": 3}
clean["AgeGroup_Encoded"] = clean["AgeGroup"].map(age_group_map)

# Combine features for ML
ml_features = pd.concat([
    clean[["Age", "Salary", "Rating", "Tenure_Days", "AgeGroup_Encoded"]],
    dept_dummies
], axis=1).dropna()

print("ML-ready feature matrix:")
print(ml_features.head())
print(f"Shape: {ml_features.shape}")

## 10. Practice Problems

In [ ]:
# Problem 1: Complete the full cleaning pipeline for this dataset.
# Steps: remove duplicates, handle missing Age (fill with median),
# cap Salary outliers at 99th percentile, standardize Department,
# parse dates, create Year/Month features, encode Department.
df_raw = messy.copy()

df_clean = df_raw.drop_duplicates(subset="ID")
df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median())
df_clean.loc[df_clean["Age"] < 0, "Age"] = df_clean["Age"].median()
salary_99 = df_clean["Salary"].quantile(0.99)
df_clean.loc[df_clean["Salary"] > salary_99, "Salary"] = salary_99
df_clean["Department"] = df_clean["Department"].str.strip().str.title()
dept_map = {"Hr": "HR", "It": "IT", "Finance": "Finance"}
df_clean["Department"] = df_clean["Department"].map(dept_map).fillna("Unknown")
df_clean["Join_Date"] = pd.to_datetime(df_clean["Join_Date"], errors="coerce")
df_clean["Year"] = df_clean["Join_Date"].dt.year
df_clean["Month"] = df_clean["Join_Date"].dt.month
df_clean = pd.get_dummies(df_clean, columns=["Department"], prefix="Dept")

print("Cleaned data:")
print(df_clean.head())
print(f"Final shape: {df_clean.shape}")

In [ ]:
# Problem 2: Detect outliers in Age using z-score. Remove rows where
# |z-score| > 2. What percentage of data is removed?
from scipy import stats

df2 = messy.drop_duplicates(subset="ID").copy()
df2["Age"] = df2["Age"].fillna(df2["Age"].median())
z = np.abs(stats.zscore(df2["Age"]))
removed_pct = (z > 2).mean() * 100
print(f"Percentage removed (|z|>2): {removed_pct:.1f}%")
print(f"Rows before: {len(df2)}, after: {len(df2[z <= 2])}")

In [ ]:
# Problem 3: The Rating column has values as strings ("4.5", "3", etc.).
# Convert to float, fill missing with mean, then discretize
# into 3 bins: 'Low' (< 3.5), 'Medium' (3.5-4.2), 'High' (> 4.2).
df3 = messy.copy()
df3["Rating"] = pd.to_numeric(df3["Rating"], errors="coerce")
df3["Rating"] = df3["Rating"].fillna(df3["Rating"].mean())
bins = [-np.inf, 3.5, 4.2, np.inf]
labels = ["Low", "Medium", "High"]
df3["Rating_Band"] = pd.cut(df3["Rating"], bins=bins, labels=labels)
print(df3[["Rating", "Rating_Band"]].dropna())

## 11. Summary Checklist – ML Data Cleaning

| Step | Technique |
|------|-----------|
| Deduplicate | `drop_duplicates()` |
| Handle missing values | `fillna()` / `dropna()` / imputation |
| Outlier detection | IQR / z-score / domain caps |
| Standardize categories | Strip, title case, map |
| Parse dates | `to_datetime(..., errors='coerce')` |
| Type coercion | `pd.to_numeric()` / `astype()` |
| Feature engineering | `dt.year`, `dt.month`, `dt.dayofweek`, tenure |
| Binning | `pd.cut()` / `pd.qcut()` |
| Encoding | `get_dummies()` / label mapping |
| Final check | `info()`, `isna().sum()`, `describe()` |